### Harmonization of the metadata of the remapped studies
- **Developed by:** Anna Maguza
- **Affilation:** Faculty of Medicine, Würzburg University
- **Creation date:** 8th of November 2024
- **Last modified date:** 8th of November 2024

This notebooks uses natural language processing (NLP) techniques to harmonize metadata columns and values. Here we process such studies:
* E-MTAB-8901 (Elementaite et al, 2021) - fetal developing gut
* E-MTAB-9536 (Elementaite et al, 2021) - fetal developing gut
* E-MTAB-9543 (Elementaite et al, 2021) - adult gut
* E-MTAB-9489 (Holloway et al, 2021) - fetal developing gut
* E-MTAB-9720 (Holloway et al, 2021) - fetal enteroids

+ import packages

In [1]:
import pandas as pd
from typing import Dict, List, Optional, Union
import os
from anthropic import Anthropic
import json

In [2]:
e8901 = pd.read_csv('raw_fastq_files/Elmentaite_2021/metadata/E-MTAB-8901.sdrf.txt', sep='\t')

In [12]:
e8901.columns

Index(['Source Name', 'Comment[ENA_SAMPLE]', 'Comment[BioSD_SAMPLE]',
       'Characteristics[organism]', 'Characteristics[developmental stage]',
       'Characteristics[age]', 'Unit[time unit]', 'Term Source REF',
       'Term Accession Number', 'Characteristics[gestational age]',
       'Unit[time unit].1', 'Term Source REF.1', 'Term Accession Number.1',
       'Characteristics[sex]', 'Characteristics[disease]',
       'Characteristics[individual]', 'Characteristics[organism part]',
       'Characteristics[cell type]', 'Characteristics[immunophenotype]',
       'Characteristics[growth condition]', 'Characteristics[passage]',
       'Material Type', 'Protocol REF', 'Protocol REF.1', 'Protocol REF.2',
       'Protocol REF.3', 'Protocol REF.4', 'Extract Name',
       'Comment[LIBRARY_LAYOUT]', 'Comment[LIBRARY_SELECTION]',
       'Comment[LIBRARY_SOURCE]', 'Comment[LIBRARY_STRAND]',
       'Comment[LIBRARY_STRATEGY]', 'Comment[NOMINAL_LENGTH]',
       'Comment[NOMINAL_SDEV]', 'Comment[O

In [3]:
e9489 = pd.read_csv('raw_fastq_files/Holloway_2021/E-MTAB-9489/E-MTAB-9489.sdrf.txt', sep='\t')

In [4]:
e9228 = pd.read_csv('raw_fastq_files/Holloway_2020/E-MTAB-9228.sdrf.txt', sep='\t')

In [5]:
e10187 = pd.read_csv('raw_fastq_files/Yu_2021/E-MTAB-10187.sdrf.txt', sep='\t')

### Trying Claude

In [13]:
import pandas as pd
from typing import Dict, List, Optional, Union
from anthropic import Anthropic
import json
import re

class MetadataHarmonizer:
    def __init__(self, api_key: str):
        """
        Initialize the MetadataHarmonizer with Anthropic API key.
        
        Parameters:
        -----------
        api_key : str
            Anthropic API key for Claude
        """
        self.anthropic = Anthropic(api_key=api_key)
        
        # Define standard fields and their expected formats
        self.standard_fields = {
            'donor_id': str,
            'sample_id': str,
            'organism': str,
            'organ': str,
            'tissue': str,
            'sex': str,
            'developmental_stage': str,
            'age': str,
            'disease': str,
            'single_cell_isolation': str,
            'library_preparation': str,
            'tenx_priming': str,
            'immunophenotype': str,
            'organoid_passage': Optional[int],
            'organoid_media': Optional[str],
            'genetic_phenotype': Optional[str]
        }
        
        # Define mapping from common input columns to standard fields
        self.column_mapping = {
            'donor_id': [
                'Characteristics[individual]',
                'Comment[BioSD_SAMPLE]'
            ],
            'sample_id': [
                'Source Name',
                'Comment[ENA_SAMPLE]'
            ],
            'organism': [
                'Characteristics[organism]'
            ],
            'organ': [
                'Characteristics[organism part]',
                'Factor Value[organism part]'
            ],
            'developmental_stage': [
                'Characteristics[developmental stage]',
                'Factor Value[developmental stage]'
            ],
            'age': [
                'Characteristics[age]',
                'Characteristics[gestational age]'
            ],
            'sex': [
                'Characteristics[sex]'
            ],
            'disease': [
                'Characteristics[disease]',
                'Factor Value[disease]'
            ],
            'single_cell_isolation': [
                'Comment[single cell isolation]',
                'Protocol REF'
            ],
            'library_preparation': [
                'Comment[library construction]',
                'Comment[LIBRARY_STRATEGY]'
            ],
            'tenx_priming': [
                'Comment[primer]',
                'Comment[end bias]'
            ],
            'immunophenotype': [
                'Characteristics[immunophenotype]',
                'Factor Value[immunophenotype]'
            ],
            'organoid_passage': [
                'Characteristics[passage]'
            ],
            'organoid_media': [
                'Characteristics[growth condition]',
                'Factor Value[growth condition]'
            ]
        }
        
    def _extract_library_info(self, row: pd.Series) -> Dict[str, str]:
        """
        Extract detailed library preparation information from various fields.
        
        Parameters:
        -----------
        row : pd.Series
            Single row of metadata
            
        Returns:
        --------
        Dict[str, str] : Dictionary containing library preparation details
        """
        library_info = {}
        
        # Extract 10X version from library construction field
        if 'Comment[library construction]' in row:
            construction = str(row['Comment[library construction]']).lower()
            if '10x' in construction:
                if 'v1' in construction:
                    library_info['version'] = '10x_v1'
                elif 'v2' in construction:
                    library_info['version'] = '10x_v2'
                elif 'v3' in construction:
                    library_info['version'] = '10x_v3'
        
        # Extract priming information
        if 'Comment[primer]' in row:
            primer = str(row['Comment[primer]']).lower()
            if '3' in primer or "3'" in primer:
                library_info['priming'] = '3_prime'
            elif '5' in primer or "5'" in primer:
                library_info['priming'] = '5_prime'
        
        # Check for UMI information
        umi_fields = ['Comment[umi barcode size]', 'Comment[umi barcode read]']
        if any(field in row.index for field in umi_fields):
            library_info['has_umi'] = True
        
        return library_info

    def _extract_age(self, row: pd.Series) -> str:
        """
        Standardize age information from various fields.
        
        Parameters:
        -----------
        row : pd.Series
            Single row of metadata
            
        Returns:
        --------
        str : Standardized age representation
        """
        age = None
        unit = None
        
        # Check regular age field
        if 'Characteristics[age]' in row:
            age = row['Characteristics[age]']
            if 'Unit[time unit]' in row:
                unit = row['Unit[time unit]']
                
        # Check gestational age field
        elif 'Characteristics[gestational age]' in row:
            age = row['Characteristics[gestational age]']
            if 'Unit[time unit].1' in row:
                unit = row['Unit[time unit].1']
        
        if pd.isna(age):
            return None
            
        # Standardize the age format
        return f"{age} {unit}" if unit else str(age)

    def _get_value_from_mapping(self, row: pd.Series, field: str) -> str:
        """
        Get value for a standard field using the column mapping.
        
        Parameters:
        -----------
        row : pd.Series
            Single row of metadata
        field : str
            Standard field name to look up
            
        Returns:
        --------
        str : Value for the field
        """
        possible_columns = self.column_mapping.get(field, [])
        
        for col in possible_columns:
            if col in row and not pd.isna(row[col]):
                return str(row[col])
        
        return None

    def harmonize_metadata(self, metadata_df: pd.DataFrame) -> pd.DataFrame:
        """
        Harmonize metadata DataFrame using column mapping and specialized extractors.
        
        Parameters:
        -----------
        metadata_df : pd.DataFrame
            Input metadata DataFrame with various column names and formats
            
        Returns:
        --------
        pd.DataFrame : Harmonized metadata with standardized columns and values
        """
        harmonized_data = []
        
        for _, row in metadata_df.iterrows():
            harmonized_record = {}
            
            # Extract basic fields using mapping
            for standard_field in self.standard_fields:
                value = self._get_value_from_mapping(row, standard_field)
                harmonized_record[standard_field] = value
            
            # Extract specialized fields
            library_info = self._extract_library_info(row)
            harmonized_record['library_preparation'] = library_info.get('version')
            harmonized_record['tenx_priming'] = library_info.get('priming')
            
            # Handle age specially
            harmonized_record['age'] = self._extract_age(row)
            
            # Clean up and standardize values using Claude
            for field, value in harmonized_record.items():
                if value is not None:
                    harmonized_record[field] = self._query_claude(
                        {field: value},
                        field
                    )
            
            harmonized_data.append(harmonized_record)
            
        return pd.DataFrame(harmonized_data)

    def _query_claude(self, metadata: Dict, field: str) -> str:
        """
        Query Claude to interpret and standardize a metadata field.
        
        Parameters:
        -----------
        metadata : Dict
            Original metadata dictionary
        field : str
            Field to interpret
            
        Returns:
        --------
        str : Standardized value for the field
        """
        prompt = f"""
        Based on this metadata: {json.dumps(metadata)}
        Please extract or infer the standardized value for: {field}
        
        Follow these rules:
        - For organism: use full scientific name (e.g., 'Homo sapiens')
        - For developmental_stage: use ['fetal_t1', 'fetal_t2', 'fetal_t3', 'child', 'adult', 'organoid']
        - For organ/tissue: use standardized anatomical terms
        - For sex: use ['male', 'female', 'unknown']
        - For library_preparation: specify version (e.g., '10x_v2', '10x_v3', 'smart-seq2')
        - For tenx_priming: use ['3_prime', '5_prime']
        
        Return only the standardized value without explanation.
        """
        
        response = self.anthropic.messages.create(
            model="claude-3-sonnet-20240229",
            max_tokens=100,
            messages=[{
                "role": "user",
                "content": prompt
            }]
        )
        
        return response.content.strip()
    
    def add_to_anndata(self, adata, harmonized_metadata: pd.DataFrame) -> None:
        """
        Add harmonized metadata to AnnData object.
        
        Parameters:
        -----------
        adata : anndata.AnnData
            AnnData object to modify
        harmonized_metadata : pd.DataFrame
            Harmonized metadata to add
        """
        for column in harmonized_metadata.columns:
            adata.obs[column] = harmonized_metadata[column]

In [16]:
api_key = ...

In [17]:
harmonizer = MetadataHarmonizer(api_key)

In [18]:
harmonized_metadata = harmonizer.harmonize_metadata(e8901)

AttributeError: 'list' object has no attribute 'strip'

In [10]:
harmonized_metadata

,donor_id,sample_id,organism,organ,tissue,sex,developmental_stage,age,disease,single_cell_isolation,library_preparation,tenx_priming,immunophenotype,organoid_passage,organoid_media,genetic_phenotype
0,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
3,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
4,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
5,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
6,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
7,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
8,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
9,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None


### Define functions

In [11]:
import pandas as pd
from collections import defaultdict
import numpy as np
from rapidfuzz import fuzz, process

class MetadataStandardizer:
    def __init__(self):
        # Common technical fields to look for
        self.tech_fields = {
            'sequencing': ['LIBRARY_STRATEGY', 'LIBRARY_SOURCE', 'LIBRARY_SELECTION', 
                         'LIBRARY_LAYOUT', 'INSTRUMENT', 'PLATFORM'],
            'protocol': ['library construction', 'single cell isolation', 
                       'spike in', 'input molecule'],
            'sample': ['organism', 'age', 'sex', 'tissue', 'disease_state'],
            '10x': ['cdna read', 'cell barcode', 'umi barcode']
        }
        
        # Common value mappings
        self.value_mappings = {
            'sex': {
                'male': ['m', 'male', 'M', 'Male'],
                'female': ['f', 'female', 'F', 'Female']
            },
            'library_type': {
                '10x': ['10x', '10X', '10xv2', '10XV2', '10x Genomics'],
                'smart-seq': ['smart-seq', 'smartseq', 'Smart-seq']
            }
        }
        
        # Columns to exclude (case-insensitive patterns)
        self.exclude_patterns = [
            'uri', 'file', 'biosd_sample', 'ref'
        ]

    def filter_columns(self, df):
        """Remove columns containing specified patterns"""
        filtered_cols = []
        for col in df.columns:
            if not any(pattern.lower() in col.lower() for pattern in self.exclude_patterns):
                filtered_cols.append(col)
        return df[filtered_cols]

    def identify_columns(self, df):
        """Identify columns by both name and content analysis"""
        column_categories = defaultdict(list)
        
        for col in df.columns:
            # Check column name
            col_lower = col.lower()
            
            # Sample a few values from the column
            sample_values = df[col].dropna().astype(str).sample(min(5, len(df))).tolist()
            
            # Analyze both column name and values
            if any(tech in col_lower for tech in ['library', 'sequencing', 'instrument']):
                column_categories['technical'].append(col)
            
            elif any(proto in col_lower for proto in ['protocol', 'method']):
                column_categories['protocol'].append(col)
                
            elif any(bio in col_lower for bio in ['organism', 'tissue', 'cell', 'disease']):
                column_categories['biological'].append(col)
                
            elif any(x in col_lower for x in ['barcode', 'umi', 'read']):
                column_categories['10x'].append(col)
                
        return column_categories

    def standardize_values(self, df, column_mapping):
        """Standardize values based on known mappings and fuzzy matching"""
        std_df = df.copy()
        
        for std_col, orig_col in column_mapping.items():
            if std_col in self.value_mappings:
                mapping = self.value_mappings[std_col]
                
                def standardize_value(val):
                    if pd.isna(val):
                        return val
                    val = str(val).lower()
                    for std_val, variants in mapping.items():
                        if val in variants:
                            return std_val
                    return val
                    
                std_df[orig_col] = std_df[orig_col].apply(standardize_value)
                
        return std_df

    def suggest_column_mapping(self, df1, df2):
        """Suggest column mappings between two dataframes"""
        mappings = {}
        
        for col1 in df1.columns:
            # Get sample values from first dataframe
            values1 = df1[col1].dropna().astype(str).sample(min(5, len(df1))).tolist()
            
            best_match = None
            best_score = 0
            
            for col2 in df2.columns:
                # Compare column names
                name_score = fuzz.ratio(col1.lower(), col2.lower())
                
                # Compare column values
                values2 = df2[col2].dropna().astype(str).sample(min(5, len(df2))).tolist()
                value_scores = []
                for v1 in values1:
                    for v2 in values2:
                        value_scores.append(fuzz.ratio(v1.lower(), v2.lower()))
                value_score = np.mean(value_scores) if value_scores else 0
                
                # Combine scores
                total_score = (name_score + value_score) / 2
                
                if total_score > 70:  # Threshold for suggesting a match
                    mappings[col1] = best_match
                    
        return mappings

    def merge_dataframes(self, dfs, on=None, how='outer'):
        """
        Merge multiple harmonized dataframes
        
        Parameters:
        -----------
        dfs : list of pandas DataFrames
            List of dataframes to merge
        on : str or list of str, optional
            Column(s) to merge on. If None, will try to merge on common columns
        how : str, default 'outer'
            Type of merge to perform ('left', 'right', 'outer', 'inner')
            
        Returns:
        --------
        pandas DataFrame
            Merged dataframe
        """
        if not dfs:
            raise ValueError("No dataframes provided for merging")
            
        # Filter out unwanted columns from all dataframes
        filtered_dfs = [self.filter_columns(df) for df in dfs]
        
        # If no merge columns specified, find common columns
        if on is None:
            common_cols = set(filtered_dfs[0].columns)
            for df in filtered_dfs[1:]:
                common_cols = common_cols.intersection(df.columns)
            if not common_cols:
                raise ValueError("No common columns found for merging")
            on = list(common_cols)
            
        # Perform the merge
        result = filtered_dfs[0]
        for df in filtered_dfs[1:]:
            result = pd.merge(result, df, on=on, how=how)
            
        return result

In [12]:
standardizer = MetadataStandardizer()

In [15]:
suggested_mappings = standardizer.suggest_column_mapping(e8901, e9489)
print("Suggested mappings:", suggested_mappings)

Suggested mappings: {'Comment[ENA_SAMPLE]': None, 'Comment[BioSD_SAMPLE]': None, 'Characteristics[organism]': None, 'Characteristics[developmental stage]': None, 'Term Source REF.1': None, 'Characteristics[sex]': None, 'Characteristics[disease]': None, 'Characteristics[organism part]': None, 'Protocol REF': None, 'Protocol REF.1': None, 'Protocol REF.2': None, 'Protocol REF.3': None, 'Protocol REF.4': None, 'Comment[LIBRARY_SELECTION]': None, 'Comment[LIBRARY_SOURCE]': None, 'Comment[LIBRARY_STRATEGY]': None, 'Comment[cdna read]': None, 'Comment[cdna read offset]': None, 'Comment[cdna read size]': None, 'Comment[cell barcode offset]': None, 'Comment[cell barcode read]': None, 'Comment[cell barcode size]': None, 'Comment[end bias]': None, 'Comment[input molecule]': None, 'Comment[library construction]': None, 'Comment[primer]': None, 'Comment[sample barcode offset]': None, 'Comment[sample barcode read]': None, 'Comment[sample barcode size]': None, 'Comment[single cell isolation]': None,

In [ ]:
# Filter and standardize each dataframe
df1_clean = standardizer.standardize_values(e8901)
df2_clean = standardizer.standardize_values(df2, column_mapping2)
df3_clean = standardizer.standardize_values(df3, column_mapping3)

# Merge all dataframes
merged_df = standardizer.merge_dataframes(
    [df1_clean, df2_clean, df3_clean],
    on=['sample_id'],  # specify common columns to merge on
    how='outer'  # or 'inner', 'left', 'right'
)

In [7]:
standardizer = MetadataStandardizer()

# Read your SDRF files
dfs = [e8901, e9489, e9228, e10187]

# Identify columns in each dataframe
for i, df in enumerate(dfs):
    categories = standardizer.identify_columns(df)
    print(f"Dataset {i+1} columns:")
    for category, cols in categories.items():
        print(f"{category}: {cols}")

# Get suggested mappings between datasets
mappings = standardizer.suggest_column_mapping(dfs[0], dfs[1])
print("\nSuggested column mappings:")
for col1, col2 in mappings.items():
    print(f"{col1} -> {col2}")

Dataset 1 columns:
biological: ['Characteristics[organism]', 'Characteristics[disease]', 'Characteristics[organism part]', 'Characteristics[cell type]', 'Comment[cell barcode offset]', 'Comment[cell barcode read]', 'Comment[cell barcode size]', 'Comment[single cell isolation]', 'Factor Value[disease]', 'Factor Value[organism part]']
protocol: ['Protocol REF', 'Protocol REF.1', 'Protocol REF.2', 'Protocol REF.3', 'Protocol REF.4', 'Protocol REF.5']
technical: ['Comment[LIBRARY_LAYOUT]', 'Comment[LIBRARY_SELECTION]', 'Comment[LIBRARY_SOURCE]', 'Comment[LIBRARY_STRAND]', 'Comment[LIBRARY_STRATEGY]', 'Comment[library construction]']
10x: ['Comment[cdna read]', 'Comment[cdna read offset]', 'Comment[cdna read size]', 'Comment[sample barcode offset]', 'Comment[sample barcode read]', 'Comment[sample barcode size]', 'Comment[umi barcode offset]', 'Comment[umi barcode read]', 'Comment[umi barcode size]', 'Comment[read1 file]', 'Comment[read2 file]']
Dataset 2 columns:
biological: ['Characteristi

In [8]:
std_df0 = standardizer.standardize_values(dfs[0], mappings)
std_df1 = standardizer.standardize_values(dfs[1], mappings)
std_df2 = standardizer.standardize_values(dfs[2], mappings)
std_df3 = standardizer.standardize_values(dfs[2], mappings)

In [9]:
std_df0

,Source Name,Comment[ENA_SAMPLE],Comment[BioSD_SAMPLE],Characteristics[organism],Characteristics[developmental stage],Characteristics[age],Unit[time unit],Term Source REF,Term Accession Number,Characteristics[gestational age],...,Comment[FASTQ_URI],Comment[read2 file],Comment[FASTQ_URI].1,Comment[index1 file],Comment[FASTQ_URI].2,Factor Value[disease],Factor Value[developmental stage],Factor Value[organism part],Factor Value[immunophenotype],Factor Value[growth condition]
0,2206_p2_WNT3A_cells,ERS4414920,SAMEA6655451,Homo sapiens,embryonic human stage,,,,,8.5,...,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,4918STDY8366756_S1_L001_R2_001.fastq.gz,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,4918STDY8366756_S1_L001_I1_001.fastq.gz,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,normal,embryonic human stage,ileum,total cells,organoid grown in conditioned medium
1,p172038_cells,ERS4414921,SAMEA6655452,Homo sapiens,embryonic human stage,,,,,7.4,...,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,4918STDY7590324_S1_L001_R2_001.fastq.gz,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,4918STDY7590324_S1_L001_I1_001.fastq.gz,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,normal,embryonic human stage,ileum,total cells,organoid grown in conditioned medium
2,p172039_cells,ERS4414922,SAMEA6655453,Homo sapiens,embryonic human stage,,,,,7.4,...,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,4918STDY7590325_S1_L001_R2_001.fastq.gz,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,4918STDY7590325_S1_L001_I1_001.fastq.gz,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,normal,embryonic human stage,ileum,total cells,organoid grown in conditioned medium
3,TIp12038_cells,ERS4414923,SAMEA6655454,Homo sapiens,embryonic human stage,,,,,8.4,...,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,4918STDY7426910_S1_L001_R2_001.fastq.gz,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,4918STDY7426910_S1_L001_I1_001.fastq.gz,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,normal,embryonic human stage,ileum,total cells,organoid grown in conditioned medium
4,TIp12039_cells,ERS4414924,SAMEA6655455,Homo sapiens,embryonic human stage,,,,,8.4,...,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,4918STDY7426911_S1_L001_R2_001.fastq.gz,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,4918STDY7426911_S1_L001_I1_001.fastq.gz,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,normal,embryonic human stage,ileum,total cells,organoid grown in conditioned medium
5,2206_p2_RSPO_cells,ERS4414925,SAMEA6655456,Homo sapiens,embryonic human stage,,,,,8.5,...,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,4918STDY8366754_S1_L001_R2_001.fastq.gz,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,4918STDY8366754_S1_L001_I1_001.fastq.gz,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,normal,embryonic human stage,ileum,total cells,organoid grown in unconditioned medium (no WNT...
6,BRC2133_CO_neg_cells,ERS4414926,SAMEA6655457,Homo sapiens,9th week post-fertilization human stage,,,,,11.9,...,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,4918STDY7717789_S1_L001_R2_001.fastq.gz,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,4918STDY7717789_S1_L001_I1_001.fastq.gz,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,normal,9th week post-fertilization human stage,colon,EPCAM negative,primary tissue
7,BRC2134_CO_neg_cells,ERS4414927,SAMEA6655458,Homo sapiens,10th week post-fertilization human stage,,,,,12,...,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,4918STDY7718974_S1_L001_R2_001.fastq.gz,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,4918STDY7718974_S1_L001_I1_001.fastq.gz,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,normal,10th week post-fertilization human stage,colon,EPCAM negative,primary tissue
8,BRC2121_DU_neg_cells,ERS4414928,SAMEA6655459,Homo sapiens,9th week post-fertilization human stage,,,,,11.2,...,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,4918STDY7693763_S1_L001_R2_001.fastq.gz,ftp://ftp.ebi.ac.uk/pub/databases/mi

### c

In [33]:
harmonizer = MetadataStandardizer()

In [34]:
test_df = pd.DataFrame({
    'Characteristics[organism]': ['Homo sapiens', 'Homo sapiens'],
    'Characteristics[organism part]': ['brain', 'heart'],
    'LibraryLayout': ['10x', 'Smart-seq'],
})


In [35]:
test_result = harmonizer.harmonize_metadata([test_df])
print("Test result:")
print(harmonizer.generate_report(test_result))


Columns in dataframe: ['Characteristics[organism]', 'Characteristics[organism part]', 'LibraryLayout']
Detected format: unknown
Column mapping: {'Characteristics[organism]': 'organism', 'Characteristics[organism part]': 'organism', 'LibraryLayout': 'library_protocol'}
Test result:
Metadata Harmonization Report

Summary Statistics:
Total Samples: 0
Unique Tissues: 0
Protocols Found: 10x, smart_seq

Technical Metadata:
Library Protocol: 10x, smart_seq

Biological Metadata:
Organism: organism


In [36]:
harmonized_data = harmonizer.harmonize_metadata([e8901, sra])
print("\nActual result:")
print(harmonizer.generate_report(harmonized_data))

Columns in dataframe: ['Source Name', 'Comment[ENA_SAMPLE]', 'Comment[BioSD_SAMPLE]', 'Characteristics[organism]', 'Characteristics[developmental stage]', 'Characteristics[age]', 'Unit[time unit]', 'Term Source REF', 'Term Accession Number', 'Characteristics[gestational age]', 'Unit[time unit].1', 'Term Source REF.1', 'Term Accession Number.1', 'Characteristics[sex]', 'Characteristics[disease]', 'Characteristics[individual]', 'Characteristics[organism part]', 'Characteristics[cell type]', 'Characteristics[immunophenotype]', 'Characteristics[growth condition]', 'Characteristics[passage]', 'Material Type', 'Protocol REF', 'Protocol REF.1', 'Protocol REF.2', 'Protocol REF.3', 'Protocol REF.4', 'Extract Name', 'Comment[LIBRARY_LAYOUT]', 'Comment[LIBRARY_SELECTION]', 'Comment[LIBRARY_SOURCE]', 'Comment[LIBRARY_STRAND]', 'Comment[LIBRARY_STRATEGY]', 'Comment[NOMINAL_LENGTH]', 'Comment[NOMINAL_SDEV]', 'Comment[ORIENTATION]', 'Comment[cdna read]', 'Comment[cdna read offset]', 'Comment[cdna rea

/tmp/ipykernel_2873424/729161748.py:181: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  sample_meta = std_df[available_cols].to_dict("records")
/tmp/ipykernel_2873424/729161748.py:181: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  sample_meta = std_df[available_cols].to_dict("records")


In [37]:
harmonized_data 

{'technical_metadata': {'library_protocol': ['library_protocol'],
  'sequencing_instrument': ['sequencing_instrument']},
 'biological_metadata': {'organism': ['homo sapiens', 'organism'],
  'cell_type': ['intestinal epithelial cell']},
 'sample_metadata': [{'sample_id': '2206_p2_WNT3A_cells',
   'library_protocol': '10xV2',
   'organism': 'ileum',
   'cell_type': '  '},
  {'sample_id': 'p172038_cells',
   'library_protocol': '10xV2',
   'organism': 'ileum',
   'cell_type': '  '},
  {'sample_id': 'p172039_cells',
   'library_protocol': '10xV2',
   'organism': 'ileum',
   'cell_type': '  '},
  {'sample_id': 'TIp12038_cells',
   'library_protocol': '10xV2',
   'organism': 'ileum',
   'cell_type': '  '},
  {'sample_id': 'TIp12039_cells',
   'library_protocol': '10xV2',
   'organism': 'ileum',
   'cell_type': '  '},
  {'sample_id': '2206_p2_RSPO_cells',
   'library_protocol': '10xV2',
   'organism': 'ileum',
   'cell_type': '  '},
  {'sample_id': 'BRC2133_CO_neg_cells',
   'library_protocol

In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, List, Any, Optional
from dataclasses import dataclass
from collections import defaultdict

@dataclass
class MetadataField:
    standard_name: str
    common_variations: List[str]
    data_type: str
    ontology: Optional[str] = None
    description: str = ""

class MetadataStandardizer:
    def __init__(self):
        self.standard_fields = {
            "library_protocol": MetadataField(
                standard_name="library_protocol",
                common_variations=["Library_Preparation_Protocol", "library construction", 
                                 "LibraryLayout", "LIBRARY_LAYOUT"],
                data_type="string",
                description="Library preparation protocol"
            ),
            "sequencing_instrument": MetadataField(
                standard_name="sequencing_instrument",
                common_variations=["Instrument", "Platform", "instrument_model"],
                data_type="string",
                description="Sequencing instrument used"
            ),
            "organism": MetadataField(
                standard_name="organism",
                common_variations=["Characteristics[organism]", "Organism", "species"],
                data_type="string",
                ontology="NCBITaxon"
            ),
            "tissue": MetadataField(
                standard_name="tissue_type",
                common_variations=["Characteristics[organism part]", "tissue", "organism_part",
                                 "Factor Value[organism part]"],
                data_type="string",
                ontology="UBERON"
            ),
            "sample_id": MetadataField(
                standard_name="sample_id",
                common_variations=["Source Name", "Sample Name", "BioSample", "sample_accession"],
                data_type="string"
            ),
            "cell_type": MetadataField(
                standard_name="cell_type",
                common_variations=["Characteristics[cell type]", "cell_type", 
                                 "Factor Value[cell type]"],
                data_type="string",
                ontology="CL"
            )
        }
        
        self.value_mappings = {
            "library_protocol": {
                "10x": ["10x", "10X", "10xv2", "10XV2", "10x Genomics"],
                "smart_seq": ["Smart-seq", "SMART-seq", "smartseq"],
            },
            "sequencing_instrument": {
                "nextseq500": ["NextSeq 500", "NEXTSEQ500", "NextSeq500"],
                "novaseq6000": ["NovaSeq 6000", "NOVASEQ6000", "NovaSeq6000"]
            }
        }

    def clean_value(self, value: Any) -> str:
        """Clean a single value"""
        if pd.isna(value):
            return ""
        return str(value).lower().strip()

    def get_unique_cleaned_values(self, values) -> List[str]:
        """Get unique cleaned values from a list or series"""
        cleaned = [self.clean_value(v) for v in values]
        return sorted(set(v for v in cleaned if v))

    def detect_format(self, df: pd.DataFrame) -> str:
        """Detect the format of the input metadata file"""
        cols = df.columns.tolist()
        print("Columns in dataframe:", cols)
        
        if "Run" in cols and "Assay Type" in cols:
            return "sra"
        elif "Source Name" in cols and any("Characteristics[" in col for col in cols):
            return "sdrf"
        else:
            return "unknown"

    def standardize_column_names(self, df: pd.DataFrame) -> pd.DataFrame:
        """Standardize column names"""
        std_df = df.copy()
        column_mapping = {}
        
        for col in df.columns:
            col_lower = col.lower()
            for field in self.standard_fields.values():
                if any(var.lower() in col_lower for var in field.common_variations):
                    column_mapping[col] = field.standard_name
                    break
        
        print("Column mapping:", column_mapping)
        return std_df.rename(columns=column_mapping)

    def standardize_values(self, df: pd.DataFrame) -> pd.DataFrame:
        """Standardize values in specific columns"""
        std_df = df.copy()
        
        for field, mappings in self.value_mappings.items():
            if field in std_df.columns:
                # Create a new standardized column
                std_values = std_df[field].copy()
                
                # Clean and standardize values
                for i, value in enumerate(std_df[field]):
                    cleaned_value = self.clean_value(value)
                    for std_name, variants in mappings.items():
                        if cleaned_value in [self.clean_value(v) for v in variants]:
                            std_values.iloc[i] = std_name
                            break
                
                std_df[field] = std_values
                
        return std_df

    def extract_metadata(self, df: pd.DataFrame, fields: List[str]) -> Dict[str, List[str]]:
        """Extract metadata for specified fields"""
        metadata = {}
        for field in fields:
            if field in df.columns:
                unique_vals = self.get_unique_cleaned_values(df[field])
                if unique_vals:
                    metadata[field] = unique_vals
        return metadata

    def extract_technical_metadata(self, df: pd.DataFrame) -> Dict[str, List[str]]:
        """Extract technical metadata"""
        return self.extract_metadata(df, ["library_protocol", "sequencing_instrument"])

    def extract_biological_metadata(self, df: pd.DataFrame) -> Dict[str, List[str]]:
        """Extract biological metadata"""
        return self.extract_metadata(df, ["organism", "tissue_type", "cell_type"])

    def harmonize_metadata(self, metadata_files: List[pd.DataFrame]) -> Dict[str, Any]:
        """Main harmonization function"""
        harmonized_data = {
            "technical_metadata": defaultdict(set),
            "biological_metadata": defaultdict(set),
            "sample_metadata": [],
            "summary": {}
        }
        
        for df in metadata_files:
            if df.empty:
                continue
                
            format_type = self.detect_format(df)
            print(f"Detected format: {format_type}")
            
            # Standardize columns and values
            std_df = self.standardize_column_names(df)
            std_df = self.standardize_values(std_df)
            
            # Extract metadata
            tech_meta = self.extract_technical_metadata(std_df)
            bio_meta = self.extract_biological_metadata(std_df)
            
            # Update metadata collections
            for k, v in tech_meta.items():
                harmonized_data["technical_metadata"][k].update(v)
            for k, v in bio_meta.items():
                harmonized_data["biological_metadata"][k].update(v)
            
            # Collect sample metadata
            if "sample_id" in std_df.columns:
                relevant_cols = ["sample_id"] + [f.standard_name for f in self.standard_fields.values()]
                available_cols = [col for col in relevant_cols if col in std_df.columns]
                if available_cols:
                    sample_meta = std_df[available_cols].to_dict("records")
                    harmonized_data["sample_metadata"].extend(sample_meta)
        
        # Convert sets to sorted lists
        harmonized_data["technical_metadata"] = {k: sorted(v) for k, v in harmonized_data["technical_metadata"].items()}
        harmonized_data["biological_metadata"] = {k: sorted(v) for k, v in harmonized_data["biological_metadata"].items()}
        
        # Generate summary
        harmonized_data["summary"] = {
            "total_samples": len(harmonized_data["sample_metadata"]),
            "unique_tissues": len(harmonized_data["biological_metadata"].get("tissue_type", [])),
            "protocols_found": sorted(harmonized_data["technical_metadata"].get("library_protocol", []))
        }
        
        return harmonized_data

    def generate_report(self, harmonized_data: Dict[str, Any]) -> str:
        """Generate a readable report"""
        report = []
        report.append("Metadata Harmonization Report")
        report.append("===========================")
        
        report.append("\nSummary Statistics:")
        report.append(f"Total Samples: {harmonized_data['summary']['total_samples']}")
        report.append(f"Unique Tissues: {harmonized_data['summary']['unique_tissues']}")
        if harmonized_data['summary']['protocols_found']:
            report.append(f"Protocols Found: {', '.join(harmonized_data['summary']['protocols_found'])}")
        
        if harmonized_data['technical_metadata']:
            report.append("\nTechnical Metadata:")
            for key, values in harmonized_data['technical_metadata'].items():
                report.append(f"{key.replace('_', ' ').title()}: {', '.join(values)}")
        
        if harmonized_data['biological_metadata']:
            report.append("\nBiological Metadata:")
            for key, values in harmonized_data['biological_metadata'].items():
                report.append(f"{key.replace('_', ' ').title()}: {', '.join(values)}")
        
        return "\n".join(report)